In [21]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch
from torch.utils.data import random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [18]:
df = pd.read_csv(
"/content/fashion-mnist_train.csv",
    engine="python",
    on_bad_lines="skip"
)

# remove broken rows
df = df.dropna()

# force numeric conversion (removes hidden corruption)
df = df.apply(pd.to_numeric, errors="coerce")
df = df.dropna()

# keep only valid FashionMNIST labels (0–9)
df = df[df["label"].between(0, 9)]

print("Final shape:", df.shape)
print("Labels:", np.unique(df["label"]))

Final shape: (60000, 785)
Labels: [0 1 2 3 4 5 6 7 8 9]


In [19]:
X = df.iloc[:, 1:].values.astype(np.float32)
y = df.iloc[:, 0].values.astype(np.int64)

# normalize pixel values (0–255 → 0–1)
X = X / 255.0

In [22]:
class FMNISTDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = FMNISTDataset(X, y)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)




In [23]:
class FMNISTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(128,64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(64, 10)   # 10 classes (VERY IMPORTANT)
        )

    def forward(self, x):
        return self.net(x)

model = FMNISTModel()
model=model.to(device)

In [24]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001,weight_decay=1e-4)

In [25]:
epochs = 25

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch_features, batch_labels in train_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        # forward
        outputs = model(batch_features)

        loss = criterion(outputs, batch_labels)

        # backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

Epoch 1/25, Loss: 483.3606
Epoch 2/25, Loss: 346.2571
Epoch 3/25, Loss: 319.2395
Epoch 4/25, Loss: 301.6985
Epoch 5/25, Loss: 290.7351
Epoch 6/25, Loss: 280.2955
Epoch 7/25, Loss: 272.6866
Epoch 8/25, Loss: 264.9687
Epoch 9/25, Loss: 259.4101
Epoch 10/25, Loss: 254.3919
Epoch 11/25, Loss: 250.3908
Epoch 12/25, Loss: 245.3947
Epoch 13/25, Loss: 242.0231
Epoch 14/25, Loss: 238.9690
Epoch 15/25, Loss: 235.6207
Epoch 16/25, Loss: 232.0660
Epoch 17/25, Loss: 230.7908
Epoch 18/25, Loss: 230.9076
Epoch 19/25, Loss: 226.6754
Epoch 20/25, Loss: 223.9833
Epoch 21/25, Loss: 220.5163
Epoch 22/25, Loss: 221.0269
Epoch 23/25, Loss: 218.7999
Epoch 24/25, Loss: 216.0721
Epoch 25/25, Loss: 217.2631


In [26]:
# Training accuracy
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch_features, batch_labels in train_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        outputs = model(batch_features)
        _, predicted = torch.max(outputs, 1)

        total += batch_labels.size(0)
        correct += (predicted == batch_labels).sum().item()

train_accuracy = 100 * correct / total
print(f"Training Accuracy: {train_accuracy:.2f}%")

Training Accuracy: 92.13%


In [27]:
# Testing accuracy
correct = 0
total = 0

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        outputs = model(batch_features)
        _, predicted = torch.max(outputs, 1)

        total += batch_labels.size(0)
        correct += (predicted == batch_labels).sum().item()

test_accuracy = 100 * correct / total
print(f"Testing Accuracy: {test_accuracy:.2f}%")

Testing Accuracy: 88.69%
